In [92]:
# Import library
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import math
from sklearn.feature_extraction.text import TfidfVectorizer

Word embedding = mengubah suatu kata menjadi angka terutama menjadi vektor

In [93]:
sentence_1 = "Sally sells shiny silver seashells by the calm seashore every sunny Saturday"
sentence_2 = "Sally stacks slippery silver seashells by the windy seashore each cloudy Sunday"

# Manual

In [94]:
token_1 = word_tokenize(sentence_1.lower())
token_2 = word_tokenize(sentence_2.lower())

In [95]:
eng_stopwords = stopwords.words('english')

In [96]:
filter_1 = [word for word in token_1 if word not in eng_stopwords]
filter_2 = [word for word in token_2 if word not in eng_stopwords]

In [97]:
merge = set(filter_1).union(set(filter_2))
print(merge)

{'seashore', 'silver', 'sunny', 'saturday', 'sally', 'slippery', 'stacks', 'shiny', 'windy', 'calm', 'seashells', 'cloudy', 'sells', 'every', 'sunday'}


In [98]:
word_dict_1 = dict.fromkeys(merge, 0)
word_dict_2 = dict.fromkeys(merge, 0)

print(word_dict_1)
print(word_dict_2)

{'seashore': 0, 'silver': 0, 'sunny': 0, 'saturday': 0, 'sally': 0, 'slippery': 0, 'stacks': 0, 'shiny': 0, 'windy': 0, 'calm': 0, 'seashells': 0, 'cloudy': 0, 'sells': 0, 'every': 0, 'sunday': 0}
{'seashore': 0, 'silver': 0, 'sunny': 0, 'saturday': 0, 'sally': 0, 'slippery': 0, 'stacks': 0, 'shiny': 0, 'windy': 0, 'calm': 0, 'seashells': 0, 'cloudy': 0, 'sells': 0, 'every': 0, 'sunday': 0}


In [99]:
for word in filter_1:
    word_dict_1[word] += 1
    
for word in filter_2:
    word_dict_2[word] += 1

In [100]:
word_dict = pd.DataFrame([word_dict_1, word_dict_2])
word_dict

,seashore,silver,sunny,saturday,sally,slippery,stacks,shiny,windy,calm,seashells,cloudy,sells,every,sunday
0,1,1,1,1,1,0,0,1,0,1,1,0,1,1,0
1,1,1,0,0,1,1,1,0,1,0,1,1,0,0,1


In [101]:
def compute_TF(word_list, terms):
    tf_dict = {}
    count_terms = len(terms)
    
    for word, value in word_list.items():
        tf_dict[word] = value / count_terms
    
    return tf_dict

In [102]:
tf_1 = compute_TF(word_dict_1, filter_1)
tf_2 = compute_TF(word_dict_2, filter_2)

tfs = pd.DataFrame([tf_1, tf_2])
tfs

,seashore,silver,sunny,saturday,sally,slippery,stacks,shiny,windy,calm,seashells,cloudy,sells,every,sunday
0,0.100000,0.100000,0.1,0.1,0.100000,0.000000,0.000000,0.1,0.000000,0.1,0.100000,0.000000,0.1,0.1,0.000000
1,0.111111,0.111111,0.0,0.0,0.111111,0.111111,0.111111,0.0,0.111111,0.0,0.111111,0.111111,0.0,0.0,0.111111


In [103]:
def compute_IDF(doc_list):
    idf_dict = {}
    idf_dict = dict.fromkeys(doc_list[0], 0)
    count_document = len(doc_list)
    
    for word, value in idf_dict.items():
        idf_dict[word] = math.log10(count_document / (float(value) + 1))
        
    return idf_dict

In [104]:
idf = compute_IDF([word_dict_1, word_dict_2])
idfs = pd.DataFrame(idf.items(), columns=["Word", "IDF"])
idfs

,Word,IDF
0,seashore,0.30103
1,silver,0.30103
2,sunny,0.30103
3,saturday,0.30103
4,sally,0.30103
5,slippery,0.30103
6,stacks,0.30103
7,shiny,0.30103
8,windy,0.30103
9,calm,0.30103


In [105]:
def compute_TFIDF(tfs, idfs):
    tfidf_dict = {}
    
    for word, value in tfs.items():
        tfidf_dict[word] = value * idfs.loc[idfs['Word'] == word, 'IDF'].values[0]
    return tfidf_dict

In [106]:
tfidf_1 = compute_TFIDF(tf_1, idfs)
tfidf_2 = compute_TFIDF(tf_2, idfs)

tfidf = pd.DataFrame([tfidf_1, tfidf_2])
tfidf

,seashore,silver,sunny,saturday,sally,slippery,stacks,shiny,windy,calm,seashells,cloudy,sells,every,sunday
0,0.030103,0.030103,0.030103,0.030103,0.030103,0.000000,0.000000,0.030103,0.000000,0.030103,0.030103,0.000000,0.030103,0.030103,0.000000
1,0.033448,0.033448,0.000000,0.000000,0.033448,0.033448,0.033448,0.000000,0.033448,0.000000,0.033448,0.033448,0.000000,0.000000,0.033448


# Library

In [109]:
doc = [sentence_1.lower(), sentence_2.lower()]
vectorizer = TfidfVectorizer(stop_words="english")
analyze = vectorizer.build_analyzer

X = vectorizer.fit_transform(doc)

X1 = X.toarray()[0]
X2 = X.toarray()[1]

print(X1)
print(X2)

[0.37729199 0.         0.26844636 0.37729199 0.26844636 0.26844636
 0.37729199 0.37729199 0.26844636 0.         0.         0.
 0.37729199 0.        ]
[0.         0.37729199 0.26844636 0.         0.26844636 0.26844636
 0.         0.         0.26844636 0.37729199 0.37729199 0.37729199
 0.         0.37729199]


In [108]:
# Cosine Similarity
from numpy.linalg import norm
cosine = np.dot(X1, X2) / (norm(X1) * norm(X2))

print(cosine)

0.288253784039277
